[![cloudevel](img/cloudevel.png)](https://cloudevel.com)

# Despliegue en *Linux* por *SSH*.

No todo destino es un servicio gestionado. En muchos equipos la aplicación todavía corre en un servidor *Linux* al que se llega por *SSH* (*Secure Shell*). Este cuaderno practica un **primer despliegue remoto controlado**: autenticación por clave, verificación del anfitrión, *systemd*, *PostgreSQL* en el propio servidor y una comprobación posterior.

**Nota sobre la máquina real.** Este es el único ejercicio del curso que **no** cabe en *GitHub Codespaces*: allí no hay *systemd*, y sin él no se puede gestionar *PostgreSQL* ni la unidad ```cd441-api.service``` como en un servidor. El destino es un **Ubuntu 24.04 completo** de cada estudiante (equipo propio, máquina virtual local o máquina virtual en la nube), preparado una vez con ```api/deploy/preparar_anfitrion.sh```. El resto del curso sigue ocurriendo en el *Codespace*.

## Objetivos.

* Preparar un *Ubuntu* 24.04 completo (Docker, *PostgreSQL*, usuario de despliegue y unidad *systemd*) sin incrustar credenciales en el repositorio.
* Leer el *workflow* ```.github/workflows/deploy-linux-ssh.yml``` y ejecutarlo contra ese destino, con un *runner* autoalojado en un repositorio **privado**, o con la alternativa de un *runner* hospedado si el anfitrión es alcanzable desde internet.
* Comprobar ```GET /health``` con ```"almacen":"postgresql"``` y entender por qué el *workflow* exige un ```known_hosts``` fijado de antemano.

## Valor por defecto de la práctica, y cómo cambiarlo.

**Valor por defecto.** El destino es el *Ubuntu* 24.04 de cada estudiante. Si esa máquina **no** es alcanzable desde internet (lo habitual en un equipo propio o en un aula), se registra en **ese mismo** *Ubuntu* un *runner* autoalojado, con un usuario propio **distinto** del usuario de despliegue y **sin** pertenecer al grupo ```docker```. El *workflow* usa ```runs-on: ${{ vars.DEPLOY_RUNNER || 'self-hosted' }}``` y entra por *SSH* como el usuario de despliegue (por defecto, ```deploy```). Así se practica *SSH* de verdad, sin abrir el puerto 22 a internet, y se separan las identidades.

Un *runner* autoalojado **solo** se usa en un repositorio **privado**. En uno público, cualquiera podría abrir un *pull request* con un *workflow* ```runs-on: self-hosted``` y ejecutar código en esa máquina. El primer paso del *job* se niega a correr en ese caso; esa guarda protege este *workflow*, no sustituye la regla. Conviene registrarlo de forma efímera (```./config.sh --ephemeral```) cuando sea posible, o al menos ejecutarlo solo durante la práctica y darlo de baja al terminar. El entorno protegido ```linux``` se limita a la rama ```main```.

El archivo vigente es ```.github/workflows/deploy-linux-ssh.yml```.

**Alternativa.** Si el anfitrión tiene dirección pública y *SSH* abierto, se define la variable ```DEPLOY_RUNNER``` del entorno ```linux``` con el valor ```ubuntu-latest``` y no hace falta un *runner* propio. Un *runner* hospedado por *GitHub* no puede usarse contra un *Codespace*; no es el caso de este destino.

## Preparación del anfitrión.

En el *Ubuntu* 24.04, una sola vez y con privilegios de administración, se ejecuta ```api/deploy/preparar_anfitrion.sh```. El guion es idempotente. Instala *Docker* (con *Compose* v2), *PostgreSQL* y *curl*; crea el usuario de despliegue y autoriza **solo** su clave *ed25519*; crea el rol y la base ```cd441``` con una contraseña aleatoria; deja *PostgreSQL* escuchando solo en local (sin *TCP* externo) y autentica el rol con ```scram-sha-256```; escribe ```/etc/cd441/api.env``` (```DATABASE_URL``` y la clave de *Flask*), legible solo por *root* y el usuario de despliegue; instala y habilita la unidad ```cd441-api.service```; y muestra la huella de la clave del servidor, que servirá para verificar ```SSH_KNOWN_HOSTS```.

Esos secretos de la base **no** pasan por *GitHub*. La *API* (dentro del contenedor) monta el *socket* de *PostgreSQL* en solo lectura.

Antes, se crea un par de claves solo para esta práctica. El archivo de la clave privada **no** se versiona.

La siguiente celda creará un par de claves solo para esta práctica.

In [ ]:
mkdir -p "$HOME/cd441-destino/ssh"
ssh-keygen -t ed25519 -f "$HOME/cd441-destino/ssh/deploy" -N ""

El guion de preparación se ejecuta **en el Ubuntu 24.04**, no en el *Codespace*, copiando antes el repositorio (o al menos ```api/deploy/```) y la clave pública. El uso es:

```bash
sudo bash api/deploy/preparar_anfitrion.sh /ruta/a/deploy.pub deploy /opt/cd441
```

**Salida típica** (fragmento final; la huella concreta cambia en cada máquina):

```text
== 5. Huella de la clave ed25519 de ESTE servidor
256 SHA256:xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx root@ubuntu (ED25519)
Compárala con: ssh-keyscan -t ed25519 -p 22 192.0.2.10 | ssh-keygen -lf -
Listo. Configura SSH_DEPLOY_DIR=/opt/cd441 y SSH_USER=deploy en GitHub.
```

En esa línea de comparación, el puerto y la dirección son un ejemplo (```192.0.2.10``` está reservada a documentación). Hay que usar el puerto y el anfitrión reales, **comparar** la huella con la que imprimió el guion en el propio servidor, y guardar el resultado como secreto ```SSH_KNOWN_HOSTS```. Nunca se ejecuta ```ssh-keyscan``` dentro del *pipeline*.


## Controles mínimos y entorno ```linux```.

Un despliegue por *SSH* no es «un comando remoto y ya». En este curso se exige:

* **Claves, no contraseñas**, y la clave privada solo como secreto de *GitHub*, nunca en el repositorio.
* **Verificación del anfitrión** con un ```known_hosts``` fijado de antemano y ```StrictHostKeyChecking=yes```.
* **Clave efímera en el *runner***: se escribe en ```$RUNNER_TEMP``` (permisos ```600```) y se borra al final, también si el *job* falla (```if: always()```). Eso importa en un *runner* autoalojado, que no se destruye tras el *job*.
* **Validación de entradas** antes de interpolarlas en comandos remotos, y un guion remoto entre comillas (```<<'REMOTO'```) para que nada se expanda en el *runner*.
* **Pasos idempotentes**: volver a ejecutar el *workflow* deja el servicio en el mismo estado deseado (```docker compose up -d --wait```).
* **Evidencia posterior**: una petición ```GET /health``` en el propio destino, comprobando ```"status":"ok"``` y que ```"version"``` sea el *SHA* del *commit*.

El *workflow* no reconstruye la imagen en el servidor: copia solo ```api/deploy/compose.yaml```, descarga de *GHCR* la imagen ```api``` de este repositorio, etiquetada con el *SHA* del *commit* (paquete público, sin inicio de sesión en el anfitrión) y la levanta con *Compose*. El manifiesto publica el puerto solo en ```127.0.0.1:8080```, deja el contenedor en solo lectura y elimina capacidades.

En el repositorio se configura el entorno protegido ```linux``` (*Settings* → *Environments*):

* secretos: ```SSH_HOST```, ```SSH_USER```, ```SSH_PORT```, ```SSH_PRIVATE_KEY``` y ```SSH_KNOWN_HOSTS```;
* variables: ```SSH_DEPLOY_DIR``` y, si aplica, ```DEPLOY_RUNNER```.

El *workflow* vigente publica, desde la integración continua, la imagen en *GHCR* y, en el destino, actualiza el manifiesto y el servicio. Las consultas a *PostgreSQL* van parametrizadas.

## El *workflow* ```deploy-linux-ssh.yml```.

El archivo vigente es ```.github/workflows/deploy-linux-ssh.yml```. Se dispara de forma manual (*workflow_dispatch*).

La siguiente celda mostrará el contenido de referencia de ese *workflow*.

```yaml
# Despliegue en un anfitrión Linux por SSH (cuaderno 17).
#
# Destino: un Ubuntu 24.04 COMPLETO de cada estudiante (con systemd y PostgreSQL),
# preparado con api/deploy/preparar_anfitrion.sh. No puede ser un Codespace:
# allí no hay systemd.
#
# Runner: por defecto `self-hosted`, porque un Ubuntu propio (en casa, en el aula,
# en una VM local) normalmente no es alcanzable desde los runners de GitHub.
# Si el anfitrión tiene IP pública y SSH abierto, define la variable de entorno
# DEPLOY_RUNNER=ubuntu-latest y no hace falta runner propio.
#   ⚠️ Runner autoalojado = solo en un repositorio PRIVADO. En uno público,
#   cualquiera puede abrir un pull request con un workflow que corra código en
#   tu máquina. Este job se niega a correr en ese caso (primer paso).
#
# Buenas prácticas que muestra este archivo:
#  - known_hosts FIJADO de antemano (secreto SSH_KNOWN_HOSTS) y
#    StrictHostKeyChecking=yes: nada de `ssh-keyscan` dentro del pipeline.
#  - Clave ed25519 dedicada, escrita en un directorio temporal y borrada al final
#    (importante en un runner autoalojado, que no se destruye tras el job).
#  - Validación estricta de entradas antes de usarlas en comandos remotos.
#  - Script remoto entre comillas ('REMOTO'): nada se expande en el runner;
#    los valores viajan como argumentos ya validados.
#  - Imagen inmutable por SHA (la que publicó el CI en GHCR); despliegue idempotente
#    con `docker compose up -d --wait` y evidencia final en GET /health.
#
# Entorno `linux` (Settings → Environments → linux):
#   Secretos: SSH_HOST, SSH_USER, SSH_PORT, SSH_PRIVATE_KEY, SSH_KNOWN_HOSTS
#   Variables: SSH_DEPLOY_DIR, DEPLOY_RUNNER (opcional: self-hosted | ubuntu-latest)
# Los secretos de la base de datos NO pasan por GitHub: viven en el anfitrión
# (/etc/cd441/api.env).
name: Despliegue en Linux por SSH

on:
  workflow_dispatch:

permissions:
  contents: read

concurrency:
  group: deploy-linux-ssh
  cancel-in-progress: false

jobs:
  desplegar:
    runs-on: ${{ vars.DEPLOY_RUNNER || 'self-hosted' }}
    timeout-minutes: 15
    environment:
      name: linux
    env:
      SSH_HOST: ${{ secrets.SSH_HOST }}
      SSH_USER: ${{ secrets.SSH_USER }}
      SSH_PORT: ${{ secrets.SSH_PORT }}
      SSH_DEPLOY_DIR: ${{ vars.SSH_DEPLOY_DIR }}
      APP_VERSION: ${{ github.sha }}
    steps:
      - name: Negarse a usar un runner autoalojado en un repositorio público
        if: runner.environment == 'self-hosted' && github.event.repository.private != true
        run: |
          echo "::error::Runner autoalojado + repositorio público = riesgo. Haz el repositorio privado o usa DEPLOY_RUNNER=ubuntu-latest."
          exit 1

      - name: Checkout
        uses: actions/checkout@3d3c42e5aac5ba805825da76410c181273ba90b1 # v7.0.1
        with:
          persist-credentials: false

      - name: Comprobar y validar secretos y variables
        env:
          SSH_PRIVATE_KEY: ${{ secrets.SSH_PRIVATE_KEY }}
          SSH_KNOWN_HOSTS: ${{ secrets.SSH_KNOWN_HOSTS }}
          REPOSITORIO: ${{ github.repository }}
        run: |
          faltan=0
          for nombre in SSH_HOST SSH_USER SSH_PRIVATE_KEY SSH_KNOWN_HOSTS SSH_DEPLOY_DIR; do
            if [ -z "${!nombre}" ]; then
              echo "::error::Falta ${nombre} en el entorno linux."
              faltan=1
            fi
          done
          [ "$faltan" -eq 0 ] || exit 1
          # Estos valores se usan en comandos remotos: solo se aceptan formatos seguros.
          [[ "${SSH_PORT:-22}" =~ ^[0-9]{1,5}$ ]] || { echo "::error::SSH_PORT debe ser numérico"; exit 1; }
          [[ "$SSH_HOST" =~ ^[A-Za-z0-9.:-]+$ ]] || { echo "::error::SSH_HOST no válido"; exit 1; }
          [[ "$SSH_USER" =~ ^[a-z_][a-z0-9_-]{0,31}$ ]] || { echo "::error::SSH_USER no válido"; exit 1; }
          [[ "$SSH_DEPLOY_DIR" =~ ^/[A-Za-z0-9._/-]+$ ]] || { echo "::error::SSH_DEPLOY_DIR debe ser una ruta absoluta simple"; exit 1; }
          [[ "$APP_VERSION" =~ ^[0-9a-f]{40}$ ]] || { echo "::error::SHA no válido"; exit 1; }
          echo "API_IMAGE=ghcr.io/${REPOSITORIO,,}/api:${APP_VERSION}" >> "$GITHUB_ENV"

      - name: Preparar clave y known_hosts (en un directorio temporal)
        env:
          SSH_PRIVATE_KEY: ${{ secrets.SSH_PRIVATE_KEY }}
          SSH_KNOWN_HOSTS: ${{ secrets.SSH_KNOWN_HOSTS }}
        run: |
          DIR_SSH="${RUNNER_TEMP}/ssh-cd441"
          install -m 700 -d "${DIR_SSH}"
          printf '%s\n' "${SSH_PRIVATE_KEY}" > "${DIR_SSH}/id_ed25519"
          chmod 600 "${DIR_SSH}/id_ed25519"
          printf '%s\n' "${SSH_KNOWN_HOSTS}" > "${DIR_SSH}/known_hosts"
          # Opciones comunes de ssh/scp para los pasos siguientes:
          # sin config del usuario, solo esta clave, y SOLO el known_hosts fijado.
          OPCIONES="-F /dev/null -i ${DIR_SSH}/id_ed25519 -o IdentitiesOnly=yes"
          OPCIONES+=" -o StrictHostKeyChecking=yes -o UserKnownHostsFile=${DIR_SSH}/known_hosts"
          OPCIONES+=" -o BatchMode=yes -o ConnectTimeout=15"
          echo "SSH_OPCIONES=${OPCIONES}" >> "$GITHUB_ENV"

      - name: Copiar el manifiesto de Compose al servidor
        run: |
          # shellcheck disable=SC2086 # SSH_OPCIONES se separa en palabras a propósito
          ssh ${SSH_OPCIONES} -p "${SSH_PORT:-22}" "${SSH_USER}@${SSH_HOST}" \
            mkdir -p "${SSH_DEPLOY_DIR}/api/deploy"
          # shellcheck disable=SC2086
          scp ${SSH_OPCIONES} -P "${SSH_PORT:-22}" api/deploy/compose.yaml \
            "${SSH_USER}@${SSH_HOST}:${SSH_DEPLOY_DIR}/api/deploy/compose.yaml"

      - name: Desplegar (idempotente) y comprobar /health
        run: |
          # El script remoto va entre comillas: se ejecuta tal cual en el servidor,
          # con los valores ya validados como argumentos ($1, $2, $3).
          # shellcheck disable=SC2086
          ssh ${SSH_OPCIONES} -p "${SSH_PORT:-22}" "${SSH_USER}@${SSH_HOST}" \
            bash -s -- "${SSH_DEPLOY_DIR}" "${API_IMAGE}" "${APP_VERSION}" <<'REMOTO'
          set -euo pipefail
          cd "$1"
          printf 'API_IMAGE=%s\nAPP_VERSION=%s\n' "$2" "$3" > .env
          docker compose -f api/deploy/compose.yaml --env-file .env pull --quiet
          docker compose -f api/deploy/compose.yaml --env-file .env up -d --wait
          docker compose -f api/deploy/compose.yaml --env-file .env ps
          cuerpo="$(curl -fsS --retry 5 --retry-all-errors --retry-delay 2 http://127.0.0.1:8080/health)"
          echo "${cuerpo}"
          grep -q "\"version\":\"$3\"" <<< "${cuerpo}" || { echo "Versión inesperada" >&2; exit 1; }
          grep -q '"status":"ok"' <<< "${cuerpo}"
          REMOTO

      - name: Borrar la clave del runner
        if: always()
        run: rm -rf "${RUNNER_TEMP}/ssh-cd441"
```

La ruta de salud es ```GET /health```. En este destino, con *PostgreSQL* en el anfitrión, una respuesta correcta incluye ```"almacen":"postgresql"```. Si la base no responde, la ruta devuelve 503 con ```"status":"error"```.

**Salida típica** de esa comprobación (el valor de ```version``` es el *SHA* del *commit* desplegado):

```json
{"almacen":"postgresql","status":"ok","version":"e8bf09b0144c8a6bee36d302008c9ae4a0a8c14e"}
```

El *SHA* del ejemplo es ilustrativo; en una ejecución real debe coincidir con el *commit* que disparó el *workflow*. Esta salida no se verificó aquí contra un anfitrión *Ubuntu* 24.04 con *GHCR* real (el contrato de código documenta una verificación equivalente en un entorno de prueba, con la imagen precargada).

## Actividad.

1. En el *Ubuntu* 24.04 propio, preparar el destino con ```preparar_anfitrion.sh``` y, si no es alcanzable desde internet, registrar ahí un *runner* autoalojado (usuario distinto de ```deploy```, fuera del grupo ```docker```, repositorio **privado**). Si se usa la alternativa, definir ```DEPLOY_RUNNER``` como ```ubuntu-latest```.
2. Verificar la huella del servidor, cargar en el entorno ```linux``` los secretos ```SSH_HOST```, ```SSH_USER```, ```SSH_PORT```, ```SSH_PRIVATE_KEY``` y ```SSH_KNOWN_HOSTS```, y las variables ```SSH_DEPLOY_DIR``` y, si aplica, ```DEPLOY_RUNNER```.
3. Tras un *CI* verde en ```main``` que haya publicado la imagen en *GHCR*, disparar el *workflow* ```Despliegue en Linux por SSH``` y comprobar que ```GET /health``` responde con ```"almacen":"postgresql"``` y la versión del *SHA*.
4. Volver a dispararlo sin cambiar el código y verificar que el resultado es el mismo (idempotencia). Comprobar que los datos sobreviven a ```systemctl restart cd441-api``` y a ```systemctl restart postgresql```.
5. Localizar en el *workflow* la guarda que impide un *runner* autoalojado en un repositorio público, el uso de ```SSH_KNOWN_HOSTS``` y el paso que borra la clave del *runner*.

## Cierre.

Con este cuaderno se cierra el tramo práctico de *cd441*: integración continua de una *API*, imagen publicada una vez en *GHCR*, un primer despliegue en *Cloud Run* con *WIF* y un primer despliegue en un *Ubuntu* 24.04 real por *SSH*, con *systemd* y *PostgreSQL*. Los cursos posteriores pueden dar esa práctica por cubierta y avanzar hacia infraestructura como código y servicios de *GCP* que aquí solo se nombraron (*Cloud SQL*, *Secret Manager*, *Compute Engine*).

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2023.</p>